# 00 · Data Exploration: What's in the Raw Files?

Before writing any cleaning code, this notebook opens every raw source and answers five questions:

1. Which column identifies each parcel (the join key)?
2. Where are **sale price** and **sale date**?
3. How is **property type** coded, and what is the code for single-family homes?
4. Is there a field marking **qualified (arm's-length)** sales vs. family transfers / $100 deeds?
5. Which **size, age, and location** fields exist?

The answers are summarized at the bottom, along with issues that Phase 2 (cleaning) must handle.

**Privacy note:** the tax roll contains owner names and mailing addresses. They are public record, but this
project doesn't need them, so they are never displayed here or carried forward.

Run `python src/download_data.py` first to populate `data/raw/`.

In [1]:
from pathlib import Path
import glob

import pandas as pd
import pyogrio
import geopandas as gpd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

## 1. Inventory of raw files

In [2]:
rows = []
for p in sorted(RAW.rglob("*")):
    if p.is_file() and p.suffix.lower() in {".txt", ".csv", ".zip", ".shp", ".pdf"} and ".gdb" not in str(p):
        rows.append({"file": str(p.relative_to(RAW)), "MB": round(p.stat().st_size / 1e6, 1)})
pd.DataFrame(rows).query("MB > 0.05")

,file,MB
0,docs/FDOR_2025_NAL_SDF_NAP_Users_Guide.pdf,0.6
3,fema/NFHL_12071C/S_BFE.shp,0.5
4,fema/NFHL_12071C/S_CST_TSCT_LN.shp,0.1
6,fema/NFHL_12071C/S_FLD_HAZ_AR.shp,136.0
7,fema/NFHL_12071C/S_FLD_HAZ_LN.shp,69.0
9,fema/NFHL_12071C/S_LABEL_LD.shp,0.7
10,fema/NFHL_12071C/S_LABEL_PT.shp,1.1
11,fema/NFHL_12071C/S_LIMWA.shp,0.3
14,fema/NFHL_12071C/S_PLSS_AR.shp,0.2
15,fema/NFHL_12071C/S_POL_AR.shp,0.4


## 2. Sale Data Files (SDF): the sales

LeePA publishes one SDF per tax roll year. The files have **no header row**, so column names come from the
LeePA layout PDF (`data/raw/docs/LeePA_SDF_layout.pdf`), which follows the Florida DOR standard.

In [3]:
SDF_COLS = [
    "file_type", "county_no", "parcel_id", "asmt_yr", "qual_cd", "vac_imp", "sale_chg_cd",
    "sale_price", "sale_yr", "sale_mo", "or_book", "or_page", "clerk_no", "sale_id",
]

def read_sdf(path):
    df = pd.read_csv(path, header=None, names=SDF_COLS, dtype=str)
    df = df.apply(lambda s: s.str.strip())
    df["src_file"] = Path(path).stem.replace("SDF", "")
    return df

sdf = pd.concat([read_sdf(f) for f in sorted((RAW / "leepa" / "sdf").glob("*.TXT"))], ignore_index=True)
sdf["sale_price"] = pd.to_numeric(sdf["sale_price"])
print(sdf.shape)
sdf.head()

(1056651, 15)


,file_type,county_no,parcel_id,asmt_yr,qual_cd,vac_imp,sale_chg_cd,sale_price,sale_yr,sale_mo,or_book,or_page,clerk_no,sale_id,src_file
0,S,46,01432201010080060,2019,01,I,NaN,850000.0,2018,10,NaN,NaN,2018000234960,3278385,2019Final
1,S,46,0143220200000002A,2019,01,I,NaN,320000.0,2019,05,NaN,NaN,2019000129718,3339362,2019Final
2,S,46,0143220200000011B,2019,99,I,NaN,320000.0,2019,09,NaN,NaN,2019000213780,3364428,2019Final
3,S,46,014322030003500A0,2019,99,I,NaN,100.0,2019,07,NaN,NaN,2019000172164,3351773,2019Final
4,S,46,014322030004100A0,2019,01,I,NaN,339900.0,2018,01,NaN,NaN,2018000028842,3218542,2019Final


### Which sales are in which file?

My first guess was that a roll year's SDF holds the *previous* year's sales. The data says otherwise:

In [4]:
pd.crosstab(sdf["src_file"], sdf["sale_yr"]).loc[:, "2018":]

sale_yr,2018,2019,2020,2021,2022,2023,2024,2025,2026
src_file,,,,,,,,,
2019Final,69883,47226,0,0,0,0,0,0,0
2020Final,0,67777,47916,0,0,0,0,0,0
2021Final,0,0,75236,80798,0,0,0,0,0
2022Final,0,0,0,113613,67287,0,0,0,0
2023Final,0,0,0,0,86361,51635,0,0,0
2024Final,0,0,0,0,0,75235,54810,0,0
2025Final,0,0,0,0,0,0,75135,48643,0
2026Preliminary,0,0,0,0,0,0,0,67503,27593


**Each file covers two calendar years**: the prior year plus sales from the current year up to the roll date.
So consecutive files **overlap**, and the same sale can appear twice. Do the duplicates differ between files?

In [5]:
key = ["sale_id", "parcel_id"]
dups = sdf[sdf.duplicated(key, keep=False)].sort_values(key + ["src_file"])
first, last = dups.groupby(key).first(), dups.groupby(key).last()
print(f"Sales appearing in 2 files: {len(first):,}   (in 3+ files: {(dups.groupby(key).size() > 2).sum()})")
pd.Series({c: (first[c].fillna("") != last[c].fillna("")).sum() if c != "sale_price"
           else (first[c] != last[c]).sum()
           for c in ["qual_cd", "sale_price", "vac_imp", "sale_chg_cd", "sale_yr", "sale_mo"]},
          name="n changed between files")

Sales appearing in 2 files: 379,111   (in 3+ files: 0)


qual_cd        99472
sale_price      1977
vac_imp          862
sale_chg_cd     7175
sale_yr           11
sale_mo          133
Name: n changed between files, dtype: int64

In [6]:
# How qualification codes change from the earlier file to the later one (top transitions)
(pd.DataFrame({"earlier": first["qual_cd"], "later": last["qual_cd"]})
   .value_counts().head(8).rename("n").reset_index())

,earlier,later,n
0,01,01,147197
1,11,11,95873
2,99,01,50408
3,99,11,28829
4,05,05,25757
5,99,05,7811
6,01,03,6417
7,98,98,3239


Most changes are code **99 ("qualification decision pending")** in the earlier file being resolved to a real
code (01 qualified, 11 disqualified, etc.) in the later file. **Phase 2 decision:** de-duplicate on
`(sale_id, parcel_id)` and keep the record from the **latest** file.

In [7]:
sales = (sdf.sort_values("src_file")
            .drop_duplicates(key, keep="last")
            .reset_index(drop=True))
print(f"{len(sdf):,} rows -> {len(sales):,} unique sales")
sales.groupby("sale_yr").size().rename("unique sales").to_frame().T

1,056,651 rows -> 677,540 unique sales


sale_yr,2018,2019,2020,2021,2022,2023,2024,2025,2026
unique sales,69881,68417,76003,115170,92197,80786,79405,68088,27593


### Sale qualification codes (arm's-length or not?)

From the DOR User's Guide (`data/raw/docs/FDOR_2025_NAL_SDF_NAP_Users_Guide.pdf`, pp. 17–18):

| Code | Meaning |
|---|---|
| **01** | Qualified arm's-length (deed examination) |
| **02** | Qualified arm's-length (documented evidence) |
| 03 | Arm's-length, but property changed significantly before Jan 1 (e.g., new construction, **disaster**) |
| 05 | Arm's-length, multi-parcel sale (price is for the whole bundle) |
| 11 | Quit claim / corrective / tax deed, minimum doc stamps (the classic **$100 deed**) |
| 12 | Bank / foreclosure related |
| 30 | Between relatives or affiliates |
| 37 | Atypical market exposure or motivation |
| 98 / 99 | Instrument error / decision **pending** |

In [8]:
qc = sales["qual_cd"].value_counts().rename("n").to_frame()
qc["median_price"] = sales.groupby("qual_cd")["sale_price"].median()
qc.head(12)

,n,median_price
qual_cd,,
01,327356,249900.0
11,238945,100.0
05,58091,386087.0
99,15464,60500.0
03,15036,200000.0
98,7850,100.0
30,3412,175049.5
19,2590,193750.0
16,2580,160000.0


In [9]:
print("Share of all sales at <= $100:", round((sales["sale_price"] <= 100).mean(), 3))
pd.crosstab(sales["qual_cd"].isin(["01", "02"]), sales["sale_price"] <= 100,
            rownames=["qualified 01/02"], colnames=["price <= $100"])

Share of all sales at <= $100: 0.341


price <= $100,False,True
qualified 01/02,,
False,119099,231075
True,327366,0


The qualification code does the heavy lifting: nominal-price deeds are coded 11, while qualified sales (01/02)
have realistic prices. Code **02** is almost unused in Lee County (10 sales); qualified sales are essentially all **01**.

Other SDF fields:
- `vac_imp`: **V**acant land vs. **I**mproved (we want `I`).
- `sale_chg_cd`: property changed between the assessment date and the sale; **5 = disaster**. That one
  matters for Ian, because a home sold after storm damage isn't the same "product" the roll describes.

In [10]:
sales["vac_imp"].value_counts().to_frame().T

vac_imp,I,V
count,447318,230222


In [11]:
pd.crosstab(sales["sale_chg_cd"].fillna("(blank)"), sales["sale_yr"]).loc[:, "2019":]

sale_yr,2019,2020,2021,2022,2023,2024,2025,2026
sale_chg_cd,,,,,,,,
,0,72998,113449,89401,76975,76897,66633,27543
(blank),66356,754,0,0,0,0,0,0
1,88,61,748,60,169,203,165,20
2,145,295,277,62,165,141,151,26
3,1374,1568,183,16,0,1,0,0
4,28,53,49,9,0,0,0,0
5,0,0,0,1068,3008,1595,730,0
7,426,274,464,1581,469,568,409,4


## 3. NAL tax roll: property characteristics

One file per roll year, a snapshot of every parcel as of **January 1**. Field list:
`data/raw/leepa/nal/NAL2025/Field_List_NAL12D8_2025.txt`.

In [12]:
PERSONAL = ["Name", "Address1", "Address2", "City", "State", "ZipCode", "FiduciaryName",
            "FiduciaryAddress1", "FiduciaryAddress2", "FiduciaryCity", "FiduciaryState", "FiduciaryZip",
            "MaritalStatus1", "SSN1", "MaritalStatus2", "SSN2", "SiteAddress1", "SiteAddress2", "Legal"]

def read_nal(year, usecols=None):
    path = next((RAW / "leepa" / "nal" / f"NAL{year}").glob(f"{year}_NAL12D8*.txt"))
    # 2019-2021 files are tab-delimited; 2022+ are quoted comma-delimited
    with open(path, encoding="latin-1") as f:
        sep = "\t" if "\t" in f.readline() else ","
    return pd.read_csv(path, sep=sep, dtype=str, encoding="latin-1", usecols=usecols)

nal = read_nal(2025)
nal = nal.drop(columns=PERSONAL)
print(nal.shape)
nal.head(3)

(565345, 73)


,CountyNumber,Strap,RollType,RollYear,DORland,ApprLand,SpecialAssmt,JustValue,ChangeJustValue,ChangeReason,SchoolAssdValue,AssessedValue,SchoolTaxableValue,TaxableValue,HomesteadJust,HomesteadAssd,NonHomesteadJust,NonHomesteadAssd,ResAndNonResJust,ResAndNonResAssd,AgJust,AgAssd,HWRJust,HWRAssd,ConvJust,...,ResidentialUnits,SpecialFeatures,DomicileState,FiduciaryType,Confidential,MarketArea,NeighborhoodCode,PublicLand,TaxingAuthority,Township,Range,Section,CensusBlockGroup,SiteCity,SiteZip,AlternateKey,PortabilityFlag,PortabilityOwners,PortabilityAmount,PortabilityCounty,PortabilityParcel,PortabilityYear,Exemptions,ParcelIDChange,SequenceNumber
0,46,014322L22010093R0,R,2025,000,0,0,156978.0,0.0,0,156978.0,72788.0,156978.0,72788.0,0.0,0.0,156978.0,72788.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,FL,0,0,25,25114000,NaN,034,43S,22E,01,120710101041,PUNTA GORDA,33955,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,552556
1,46,01432400000040000,R,2025,0N.,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,FL,0,0,20,20110000,NaN,000,43S,24E,01,120710202011,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,507731
2,46,01432500000030000,R,2025,068,0,0,429677.0,0.0,0,294022.0,294022.0,294022.0,294022.0,0.0,0.0,0.0,0.0,286039.0,286039.0,143638.0,7983.0,0.0,0.0,0.0,...,1.0,R2;96789;R3;4670,FL,0,0,14,14122000,NaN,202,43S,25E,01,120710301003,NORTH FORT MYERS,33917,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,218508


### Property type: `DORland`

The DOR land use code. From the User's Guide: **001 = Single Family**, 000 = vacant residential,
002 = mobile home, 004 = condominium, 008 = multi-family (<10 units).

In [13]:
nal["DORland"].value_counts().head(10).to_frame().T

DORland,001,000,004,002,008,009,0H.,010,080,048
count,268404,130488,84841,14641,11237,9166,7412,6436,6234,3551


In [14]:
sf = nal[nal["DORland"] == "001"].copy()
num_cols = ["LivingArea", "ActYearBuilt", "EffYearBuilt", "LandUnits", "JustValue", "LandValue", "BuildingCount"]
sf[num_cols] = sf[num_cols].apply(pd.to_numeric)
print(f"{len(sf):,} single-family parcels on the 2025 roll")
sf[num_cols].describe().round(0)

268,404 single-family parcels on the 2025 roll


,LivingArea,ActYearBuilt,EffYearBuilt,LandUnits,JustValue,LandValue,BuildingCount
count,268404.0,268404.0,268404.0,268404.0,268404.0,268404.0,268404.0
mean,3373.0,1998.0,2002.0,205.0,460185.0,143625.0,1.0
std,1638.0,49.0,48.0,2821.0,559033.0,325080.0,0.0
min,0.0,0.0,0.0,0.0,7278.0,7150.0,0.0
25%,2254.0,1987.0,1993.0,1.0,245237.0,39568.0,1.0
50%,2951.0,2003.0,2004.0,1.0,325166.0,74290.0,1.0
75%,4151.0,2014.0,2014.0,1.0,491128.0,132333.0,1.0
max,36151.0,2024.0,2025.0,270261.0,29973800.0,19311876.0,3.0


In [15]:
# Categorical location / quality fields for single-family parcels
pd.DataFrame({c: [sf[c].nunique(), sf[c].isna().mean().round(3), sf[c].value_counts().index[0]]
              for c in ["SiteCity", "SiteZip", "MarketArea", "NeighborhoodCode", "CensusBlockGroup",
                        "ImprovedQuality", "LandUnitCode", "DisasterCode", "DisasterYear"]},
             index=["n_unique", "share_missing", "most_common"]).T

,n_unique,share_missing,most_common
SiteCity,27,0.0,CAPE CORAL
SiteZip,40,0.0,33914
MarketArea,38,0.0,33
NeighborhoodCode,2119,0.0,1148000
CensusBlockGroup,554,0.0,120710402061
ImprovedQuality,6,0.001,3
LandUnitCode,5,0.0,5
DisasterCode,2,0.0,0
DisasterYear,3,0.0,0


Notes:
- `LivingArea` = heated/usable sq ft. `ActYearBuilt` = year built; `EffYearBuilt` = "effective" year,
  adjusted for renovations.
- `LandUnitCode` 5 = "per lot" (so `LandUnits` is mostly "1 lot"), not square feet; land size is better taken from GIS acreage.
- Some single-family parcels have `LivingArea = 0` or `ActYearBuilt = 0` (data gaps or homes destroyed / not yet built). Phase 2 must handle them.
- `CensusBlockGroup` (12 digits: state 12 + county 071 + tract + block group) gives a free join to Census ACS.
- `DisasterCode` / `DisasterYear` flag parcels with disaster value reductions: **2022 (Ian)** and **2024
  (Helene/Milton)**. The 2024 storms matter for the DiD because they hit the post-Ian period too.

In [16]:
pd.crosstab(sf["DisasterYear"], sf["DisasterCode"])

DisasterCode,0,2
DisasterYear,,
0,254986,0
2022,0,2505
2024,0,10913


In [17]:
# Disaster flags on the roll right after Ian (Jan 1, 2023 snapshot)
n23 = read_nal(2023, usecols=["Strap", "DORland", "DisasterCode", "DisasterYear"])
pd.crosstab(n23.loc[n23["DORland"] == "001", "DisasterYear"], n23.loc[n23["DORland"] == "001", "DisasterCode"])

DisasterCode,0,2
DisasterYear,,
0,233624,0
2022,0,14713


## 4. Join keys: can we link sales -> characteristics -> map?

- SDF `parcel_id` and NAL `Strap` are both 17-character parcel IDs ("STRAP" = Section-Township-Range-Area-Block-Lot).
- The GIS parcel layer also has `STRAP`.

In [18]:
nal_straps = {y: set(read_nal(y, usecols=["Strap"])["Strap"]) for y in range(2019, 2026)}

match = []
for y in range(2019, 2026):
    s = sales[sales["sale_yr"] == str(y)]
    same = s["parcel_id"].isin(nal_straps[y])
    nxt = s["parcel_id"].isin(nal_straps.get(y + 1, set()))
    match.append({"sale_yr": y, "n_sales": len(s),
                  "in NAL same year": same.mean().round(4),
                  "in same OR next year": (same | nxt).mean().round(4),
                  "in NAL 2025": s["parcel_id"].isin(nal_straps[2025]).mean().round(4)})
pd.DataFrame(match)

,sale_yr,n_sales,in NAL same year,in same OR next year,in NAL 2025
0,2019,68417,0.9932,1.0000,0.8100
1,2020,76003,0.9934,1.0000,0.8022
2,2021,115170,0.9764,1.0000,0.7733
3,2022,92197,0.9187,1.0000,0.7743
4,2023,80786,0.9021,1.0000,0.8678
5,2024,79405,0.9286,1.0000,0.9488
6,2025,68088,0.9857,0.9857,0.9857


The **NAL of the sale year** describes the property as of Jan 1 of the year it sold, i.e. *before* the sale.
It matches 90–99% of sales. The misses are parcels **created during the year** (lot splits, new subdivisions),
which don't exist yet on Jan 1. Every one of them appears on the **next** year's roll, mostly as vacant land
or brand-new single-family homes (the 2021–2024 construction boom is why the match dips in those years).

Matching everything to the latest (2025) roll is worse: it loses parcels later split or renumbered, and it
would describe post-Ian rebuilt homes for pre-Ian sales.

**Phase 2 decision:** join on `(parcel_id, sale_yr) -> (Strap, roll year)`; if missing, fall back to the
next year's roll and flag it (`nal_fallback = True`). 2025 sales have no 2026 final roll yet, so the fallback
isn't available for them.

## 5. Lee County GIS parcels: location + extra features

The NAL has no coordinates. The county's parcel geodatabase supplies a point per parcel plus some fields the
NAL lacks: bedrooms, bathrooms, pool, boat dock, seawall, and **`LANDISON`** (waterfront type).
This is a snapshot (Feb 2024), so its features reflect the current home, not necessarily the home at sale time.

In [19]:
GIS = next(RAW.glob("lee_gis/parcels/**/*.gdb"))
print(pyogrio.list_layers(GIS))
GIS_COLS = ["STRAP", "DORCODE", "LANDISON", "GISACRES", "BEDROOMS", "BATHROOMS", "POOL", "BOATDOCK",
            "SEAWALL", "GARAGE", "MAXSTORIES", "HEATEDAREA", "SITECITY", "SITEZIP"]
pts = gpd.read_file(GIS, layer="ParcelPointsData", columns=GIS_COLS)
print(pts.shape, pts.crs)
pts.drop(columns="geometry").head()

[['ParcelData' 'MultiPolygon']
 ['ParcelPointsData' 'Point']]


(555764, 15) EPSG:2237


,STRAP,DORCODE,GISACRES,LANDISON,SITECITY,SITEZIP,HEATEDAREA,MAXSTORIES,BEDROOMS,BATHROOMS,GARAGE,POOL,BOATDOCK,SEAWALL
0,044527120000D0700,01,0.0380,NaN,LEHIGH ACRES,33936,1357.0,1.0,2.0,2.0,Y,NaN,NaN,NaN
1,15462510001330090,01,0.2522,NaN,FORT MYERS,33967,1716.0,1.0,3.0,2.0,Y,NaN,NaN,NaN
2,284527L0030110020,01,0.2296,NaN,LEHIGH ACRES,33974,1458.0,1.0,3.0,2.0,Y,NaN,NaN,NaN
3,324423C3059480300,01,0.2300,NaN,CAPE CORAL,33914,2027.0,1.0,4.0,2.0,Y,Y,NaN,NaN
4,234527L1020070100,00,0.3264,NaN,LEHIGH ACRES,33974,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
pts_sf = pts[pts["DORCODE"] == "01"]
print(f"{len(pts_sf):,} single-family parcel points")
print("Share of 2025 NAL single-family parcels found in GIS:", round(sf["Strap"].isin(pts["STRAP"]).mean(), 3))
pts_sf["LANDISON"].fillna("(not waterfront)").value_counts().to_frame().T

256,045 single-family parcel points


Share of 2025 NAL single-family parcels found in GIS: 0.912


LANDISON,(not waterfront),LAKE,CANAL,GOLF,RIVER,BAY,CREEK,GULF
count,172544,37394,29810,11888,1658,1154,853,744


In [21]:
pts_sf[["BEDROOMS", "BATHROOMS", "POOL", "BOATDOCK", "SEAWALL", "GISACRES"]].describe().round(2)

,BEDROOMS,BATHROOMS,GISACRES
count,255443.00,255443.00,256045.00
mean,3.07,2.25,0.33
std,0.68,0.65,0.63
min,0.00,0.00,0.01
25%,3.00,2.00,0.20
50%,3.00,2.00,0.23
75%,3.00,2.50,0.28
max,42.00,11.00,76.16


About 9% of 2025 single-family parcels are missing from the GIS snapshot, mostly homes on parcels created
after Feb 2024. For sales up to 2024 this matters less, but Phase 2 should report the match rate by sale year.

## 6. Flood zones: two maps

- **FEMA NFHL** (current effective): `S_FLD_HAZ_AR.shp`
- **2008 FIRM** (Lee County GIS copy): the map in effect **when Ian hit**. New FIRMs took effect
  **Nov 17, 2022**, seven weeks after the storm.

`SFHA_TF = "T"` marks the **Special Flood Hazard Area** (zones A, AE, AH, AO, V, VE), the "high-risk" zone
where federally backed mortgages require flood insurance. That's the natural treatment definition.

In [22]:
nfhl = gpd.read_file(RAW / "fema/NFHL_12071C/S_FLD_HAZ_AR.shp", columns=["FLD_ZONE", "ZONE_SUBTY", "SFHA_TF"])
firm08 = gpd.read_file(RAW / "lee_gis/firm2008/FIRM.gdb", layer="FIRM", columns=["FLD_ZONE", "SFHA_TF"])
print("NFHL:", nfhl.shape, nfhl.crs.to_string(), "| 2008 FIRM:", firm08.shape, firm08.crs.to_string())
pd.concat({"NFHL current": nfhl["FLD_ZONE"].value_counts(), "FIRM 2008": firm08["FLD_ZONE"].value_counts()}, axis=1)

NFHL: (13608, 4) EPSG:4269 | 2008 FIRM: (6966, 3) EPSG:2237


/Users/lukecrater/Lee-County-Home-Prices-What-Drives-Value-and-What-Did-Hurricane-Ian-Change-/.venv/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


,NFHL current,FIRM 2008
FLD_ZONE,,
X,10800.0,3480.0
AE,1955.0,2255.0
AH,493.0,NaN
VE,236.0,103.0
A,111.0,104.0
AO,11.0,NaN
D,1.0,NaN
OPEN WATER,1.0,1.0
0.2 PCT ANNUAL CHANCE FLOOD HAZARD (X),NaN,1023.0


The layers use different coordinate systems (NFHL in lat/lon NAD83, the county layers in Florida State Plane
West feet, EPSG:2237), so everything is projected to EPSG:2237 before joining.

**Test:** place every single-family parcel point in each map and compare.

In [23]:
def flood_join(points, zones, label):
    z = zones.to_crs(points.crs)[["FLD_ZONE", "SFHA_TF", "geometry"]]
    j = gpd.sjoin(points[["STRAP", "geometry"]], z, how="left", predicate="within")
    # A point on a shared polygon edge can match twice; keep the higher-risk match
    j = j.sort_values("SFHA_TF", ascending=False).drop_duplicates("STRAP")
    return j.set_index("STRAP")[["FLD_ZONE", "SFHA_TF"]].add_suffix(f"_{label}")

fz = flood_join(pts_sf, firm08, "2008").join(flood_join(pts_sf, nfhl, "now"))
print("Share in high-risk SFHA:  2008 map =", round((fz["SFHA_TF_2008"] == "T").mean(), 3),
      "| current map =", round((fz["SFHA_TF_now"] == "T").mean(), 3))
pd.crosstab(fz["SFHA_TF_2008"].fillna("no polygon"), fz["SFHA_TF_now"].fillna("no polygon"),
            rownames=["SFHA (2008 map)"], colnames=["SFHA (current map)"], margins=True)

Share in high-risk SFHA:  2008 map = 0.344 | current map = 0.33


SFHA (current map),F,T,All
SFHA (2008 map),,,
F,159104,8873,167977
T,12397,75669,88066
no polygon,0,1,1
All,171501,84543,256044


This table is the key fact for the DiD. A meaningful number of homes **changed flood-zone status** with the
Nov 2022 remap. Defining treatment with the current map would mix "flooded/high-risk before Ian" with "newly
mapped high-risk after Ian." **Phase 5 decision:** treatment = SFHA on the **2008** map (pre-storm); the remap
can be a robustness check or its own treatment.

In [24]:
pd.crosstab(fz["FLD_ZONE_2008"].fillna("none"), fz["FLD_ZONE_now"].fillna("none"))

FLD_ZONE_now,A,AE,AH,AO,VE,X
FLD_ZONE_2008,,,,,,
0.2 PCT ANNUAL CHANCE FLOOD HAZARD (X),0,5496,140,0,0,25040
A,1139,0,0,0,0,1
AE,0,71395,830,14,316,12373
VE,0,1493,0,47,435,23
X,142,2309,786,0,0,134064
none,0,1,0,0,0,0


## 7. FRED 30-year mortgage rate

In [25]:
fred = pd.read_csv(RAW / "fred/MORTGAGE30US.csv", parse_dates=["observation_date"])
fred[fred["observation_date"] >= "2019-01-01"].set_index("observation_date")["MORTGAGE30US"].resample("YE").agg(["min", "mean", "max"]).round(2)

,min,mean,max
observation_date,,,
2019-12-31,3.49,3.94,4.51
2020-12-31,2.66,3.11,3.72
2021-12-31,2.65,2.96,3.18
2022-12-31,3.22,5.34,7.08
2023-12-31,6.09,6.81,7.79
2024-12-31,6.08,6.72,7.22
2025-12-31,6.15,6.60,7.04
2026-12-31,5.98,6.43,7.28


Rates roughly doubled in 2022, the same year as Ian. That's why the DiD needs a **comparison group**: a simple
before/after comparison would blame Ian for the rate shock, but rates hit flood-zone and non-flood-zone homes
alike, so the DiD differences them out.

*Census ACS is not downloaded yet (requires a free API key; see `src/download_data.py --only census`).
Tract-level joins will use the NAL `CensusBlockGroup` field.*

## 8. Rough sample size for the project scope

In [26]:
nal_dor = pd.concat([read_nal(y, usecols=["Strap", "DORland"]).assign(roll_yr=y) for y in range(2019, 2026)])
nal_dor = nal_dor.set_index(["Strap", "roll_yr"])["DORland"]

sy = sales["sale_yr"].astype(int)
same = pd.MultiIndex.from_arrays([sales["parcel_id"], sy])
nxt = pd.MultiIndex.from_arrays([sales["parcel_id"], sy + 1])
s = sales.assign(DORland=nal_dor.reindex(same).to_numpy())
s["nal_fallback"] = s["DORland"].isna()
s["DORland"] = s["DORland"].fillna(pd.Series(nal_dor.reindex(nxt).to_numpy(), index=s.index))
scope = s[s["sale_yr"].between("2019", "2025")]
steps = {
    "all sales 2019-2025": scope,
    "+ single family (DORland 001)": (s1 := scope[scope["DORland"] == "001"]),
    "+ improved (vac_imp I)": (s2 := s1[s1["vac_imp"] == "I"]),
    "+ qualified (qual_cd 01/02)": (s3 := s2[s2["qual_cd"].isin(["01", "02"])]),
}
funnel = pd.DataFrame({"rows": {k: len(v) for k, v in steps.items()}})
funnel["removed"] = -funnel["rows"].diff()
funnel

,rows,removed
all sales 2019-2025,580066,NaN
+ single family (DORland 001),206509,373557.0
+ improved (vac_imp I),204028,2481.0
+ qualified (qual_cd 01/02),106993,97035.0


In [27]:
s3.groupby("sale_yr").agg(n=("sale_price", "size"), from_next_yr_roll=("nal_fallback", "sum"),
                          median_price=("sale_price", "median"))

,n,from_next_yr_roll,median_price
sale_yr,,,
2019,14706,21,249900.0
2020,16890,37,280000.0
2021,21063,86,340000.0
2022,15495,746,400000.0
2023,12912,1272,400000.0
2024,12597,1128,399000.0
2025,13330,0,390000.0


## 9. Answers

**1. Parcel identifier.** A 17-character **STRAP** ID: `parcel_id` in the SDF, `Strap` in the NAL, `STRAP`
in the GIS layer. Sales join to the NAL of their sale year at 90–99%, and to the same-or-next-year roll at ~100%
(the misses are newly created parcels). GIS (a Feb 2024 snapshot) covers ~91% of 2025 single-family parcels.

**2. Sale price and date.** In the SDF: `sale_price` (from documentary stamps), `sale_yr` and `sale_mo`.
There is **no sale day**, only month. Each SDF holds two years of sales and consecutive files overlap,
so sales must be de-duplicated on `(sale_id, parcel_id)`, keeping the latest file.

**3. Property type.** NAL `DORland` (3-digit DOR land use code): **`001` = Single Family**. The GIS layer uses
the 2-digit version, `DORCODE = "01"`.

**4. Qualified sales.** SDF `qual_cd`: **`01`** (and the rarely used `02`) = qualified arm's-length.
`11` = quit-claim / $100 deeds, `30` = family transfers, `12` = bank sales, `99` = pending. `vac_imp` separates
vacant-land from improved sales; `sale_chg_cd = 5` flags disaster changes.

**5. Size, age, location.**
- *Size:* `LivingArea` (NAL); `GISACRES`, `BEDROOMS`, `BATHROOMS`, `MAXSTORIES` (GIS)
- *Age:* `ActYearBuilt`, `EffYearBuilt` (NAL)
- *Location:* `SiteCity`, `SiteZip`, `MarketArea`, `NeighborhoodCode`, `CensusBlockGroup` (NAL); parcel
  point coordinates (GIS)
- *Waterfront / amenities:* `LANDISON` (canal, gulf, bay, river, lake, golf), `POOL`, `BOATDOCK`, `SEAWALL` (GIS)
- *Flood risk:* point-in-polygon against the 2008 FIRM and current NFHL (`FLD_ZONE`, `SFHA_TF`)

### Issues to handle in Phase 2
1. **Overlapping SDF files**: de-duplicate, keep the latest record (resolves "99 pending" codes).
2. **Multi-parcel sales** (code 05): the price covers several parcels; exclude them.
3. **Time-of-sale characteristics**: use the NAL from the sale year (next year's roll for new parcels), not the latest roll.
4. **GIS snapshot is Feb 2024**: amenities like pool/dock are current-state; parcels created later won't match.
5. **Outliers / gaps**: qualified sales can still have extreme prices, `LivingArea = 0`, or `ActYearBuilt = 0`;
   needs price-per-sq-ft trimming and a rule for zeros.
6. **2024 hurricanes (Helene, Milton)** contaminate the post-Ian period; consider ending the DiD window
   in Sept 2024 or controlling for it.
7. **Month-level dates only**: Ian (Sept 28) means September 2022 is almost entirely pre-storm; treat Oct 2022+
   as post.